# CyberSentry — 01 · Data Cleaning (CIC-IDS2017)

**Project:** Intelligent Cybersecurity Threat Analysis Using ML, DL and Generative AI
**Stage:** Data loading + cleaning (before feature selection and modelling)

**Input:** the 8 original CSVs of CIC-IDS2017 (*MachineLearningCSV* version, UNB Canadian Institute for Cybersecurity) in `data/raw/`
**Output:** one cleaned file `data/processed/cicids2017_clean.parquet`

What this notebook does:
1. Loads the 8 CSVs one at a time (memory-efficient) and merges them
2. Fixes known dataset problems: spaces in column names, garbled Web Attack labels, a duplicated column, empty rows, infinite values, missing values, duplicate rows
3. Reports constant columns (listed only, not dropped — team decision)
4. Adds `binary_label` (0 = BENIGN, 1 = attack) for the ML anomaly-detection stage, and keeps the multi-class `Label` for the DL stage
5. Logs rows removed at every step and label counts before/after, for the project report

> Raw files in `data/raw/` are never modified.

## 1. Setup

In [52]:
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

# Works whether the notebook is run from the project root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
RAW_DIR = ROOT / "data" / "raw"
PROC_DIR = ROOT / "data" / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)
OUT_FILE = PROC_DIR / "cicids2017_clean.parquet"

csv_files = sorted(RAW_DIR.glob("*.csv"))
print(f"Project root : {ROOT.resolve()}")
print(f"Found {len(csv_files)} CSV files in {RAW_DIR}")
for f in csv_files:
    print(f"  {f.name:<70} {f.stat().st_size / 1e6:8.1f} MB")
assert len(csv_files) == 8, "Expected 8 CIC-IDS2017 CSV files in data/raw/"

Project root : D:\Engineering\SEM 5\MINI PROJECT II\Cybersentry
Found 8 CSV files in d:\Engineering\SEM 5\MINI PROJECT II\Cybersentry\data\raw
  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv                           77.1 MB
  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv                       76.9 MB
  Friday-WorkingHours-Morning.pcap_ISCX.csv                                  58.3 MB
  Monday-WorkingHours.pcap_ISCX.csv                                         176.9 MB
  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv                83.1 MB
  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv                     52.0 MB
  Tuesday-WorkingHours.pcap_ISCX.csv                                        135.1 MB
  Wednesday-workingHours.pcap_ISCX.csv                                      225.2 MB


## 2. Helper functions

- **`fix_label`** — In the original files the Web Attack labels contain a broken dash character (byte `0x96`), so they show up as `Web Attack � Brute Force`. We normalise them to `Web Attack - Brute Force`, `Web Attack - XSS`, `Web Attack - Sql Injection`.
- **`downcast`** — The CSVs load as 64-bit numbers. Converting to 32-bit roughly halves memory use, which matters with ~2.8 million rows. Float values are kept as `float32` (enough precision for ML); integers are only downcast when every value fits.
- **`short_name`** — a short "day" tag for each file, stored in a `source_file` column so we always know where a row came from.

In [53]:
WEB_ATTACK_RE = re.compile(r"^Web Attack\s*[^A-Za-z0-9]*\s*", flags=re.IGNORECASE)

def fix_label(label: str) -> str:
    label = str(label).strip()
    if label.lower().startswith("web attack"):
        label = WEB_ATTACK_RE.sub("Web Attack - ", label)
    return label

def downcast(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.columns:
        dt = df[col].dtype
        if pd.api.types.is_float_dtype(dt):
            df[col] = df[col].astype(np.float32)
        elif pd.api.types.is_integer_dtype(dt):
            df[col] = pd.to_numeric(df[col], downcast="integer")
    return df

def short_name(path: Path) -> str:
    return path.name.replace(".pcap_ISCX.csv", "").replace("-WorkingHours", "").replace("-workingHours", "")

## 3. Load and merge the 8 CSVs

Each file is read, cleaned at the column level and downcast **before** merging, so we never hold all the raw 64-bit data in memory at once.

The files are read with `latin-1` encoding because the Web Attack file contains a non-UTF-8 byte that makes the default reader crash.

In [54]:
frames = []
load_log = []
t0 = time.time()

for path in csv_files:
    df = pd.read_csv(path, encoding="latin-1", low_memory=False)
    df.columns = df.columns.str.strip()                     # " Label" -> "Label"
    rows_read = len(df)
    df = df.dropna(how="all")                               # fully empty rows (Thursday file has many)
    df = downcast(df)
    df["source_file"] = short_name(path)
    frames.append(df)
    load_log.append({"file": short_name(path), "rows_read": rows_read,
                     "empty_rows_dropped": rows_read - len(df), "columns": df.shape[1] - 1})
    print(f"Loaded {short_name(path):<35} {rows_read:>9,} rows")

data = pd.concat(frames, ignore_index=True)
del frames

print(f"\nMerged shape: {data.shape}  |  memory: {data.memory_usage(deep=True).sum() / 1e9:.2f} GB  |  {time.time() - t0:.0f}s")
pd.DataFrame(load_log)

Loaded Friday-Afternoon-DDos                 225,745 rows
Loaded Friday-Afternoon-PortScan             286,467 rows
Loaded Friday-Morning                        191,033 rows
Loaded Monday                                529,918 rows
Loaded Thursday-Afternoon-Infilteration      288,602 rows
Loaded Thursday-Morning-WebAttacks           170,366 rows
Loaded Tuesday                               445,909 rows
Loaded Wednesday                             692,703 rows

Merged shape: (2830743, 80)  |  memory: 0.82 GB  |  41s


,file,rows_read,empty_rows_dropped,columns
0,Friday-Afternoon-DDos,225745,0,79
1,Friday-Afternoon-PortScan,286467,0,79
2,Friday-Morning,191033,0,79
3,Monday,529918,0,79
4,Thursday-Afternoon-Infilteration,288602,0,79
5,Thursday-Morning-WebAttacks,170366,0,79
6,Tuesday,445909,0,79
7,Wednesday,692703,0,79


## 4. First look at the merged data

In [55]:
print("Columns:", data.shape[1])
print(list(data.columns))
data.head()

Columns: 80
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'EC

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Min Packet Length,Max Packet Length,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWE Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Fwd Header Length.1,Fwd Avg Bytes/Bulk,Fwd Avg Packets/Bulk,Fwd Avg Bulk Rate,Bwd Avg Bytes/Bulk,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init_Win_bytes_forward,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,source_file
0,54865,3,2,0,12,0,6,6,6.0,0.0,0,0,0.0,0.0,4.000000e+06,666666.687500,3.0,0.0,3,3,3,3.0,0.0,3,3,0,0.0,0.0,0,0,0,0,0,0,40,0,666666.687500,0.000000,6,6,6.0,0.0,0.0,0,0,0,0,1,0,0,0,0,9.0,6.0,0.0,40,0,0,0,0,0,0,2,12,0,0,33,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,Friday-Afternoon-DDos
1,55054,109,1,1,6,6,6,6,6.0,0.0,6,6,6.0,0.0,1.100917e+05,18348.623047,109.0,0.0,109,109,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,20,20,9174.311523,9174.311523,6,6,6.0,0.0,0.0,0,0,0,0,1,1,0,0,1,9.0,6.0,6.0,20,0,0,0,0,0,0,1,6,1,6,29,256,0,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,Friday-Afternoon-DDos
2,55055,52,1,1,6,6,6,6,6.0,0.0,6,6,6.0,0.0,2.307692e+05,38461.539062,52.0,0.0,52,52,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,20,20,19230.769531,19230.769531,6,6,6.0,0.0,0.0,0,0,0,0,1,1,0,0,1,9.0,6.0,6.0,20,0,0,0,0,0,0,1,6,1,6,29,256,0,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,Friday-Afternoon-DDos
3,46236,34,1,1,6,6,6,6,6.0,0.0,6,6,6.0,0.0,3.529412e+05,58823.531250,34.0,0.0,34,34,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,20,20,29411.765625,29411.765625,6,6,6.0,0.0,0.0,0,0,0,0,1,1,0,0,1,9.0,6.0,6.0,20,0,0,0,0,0,0,1,6,1,6,31,329,0,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,Friday-Afternoon-DDos
4,54863,3,2,0,12,0,6,6,6.0,0.0,0,0,0.0,0.0,4.000000e+06,666666.687500,3.0,0.0,3,3,3,3.0,0.0,3,3,0,0.0,0.0,0,0,0,0,0,0,40,0,666666.687500,0.000000,6,6,6.0,0.0,0.0,0,0,0,0,1,0,0,0,0,9.0,6.0,0.0,40,0,0,0,0,0,0,2,12,0,0,32,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,Friday-Afternoon-DDos


## 5. Fix label names

Label values are stripped of extra spaces and the garbled Web Attack names are repaired. Below: label counts **before cleaning** (raw, after only fixing names).

In [56]:
raw_unique = sorted(data["Label"].astype(str).unique())
data["Label"] = data["Label"].map(fix_label)
print("Labels before fix:", raw_unique)
print("Labels after fix :", sorted(data["Label"].unique()))

labels_before = data["Label"].value_counts()
labels_before.to_frame("count_before")

Labels before fix: ['BENIGN', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'Heartbleed', 'Infiltration', 'PortScan', 'SSH-Patator', 'Web Attack ï¿½ Brute Force', 'Web Attack ï¿½ Sql Injection', 'Web Attack ï¿½ XSS']
Labels after fix : ['BENIGN', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'Heartbleed', 'Infiltration', 'PortScan', 'SSH-Patator', 'Web Attack - Brute Force', 'Web Attack - Sql Injection', 'Web Attack - XSS']


,count_before
Label,
BENIGN,2273097
DoS Hulk,231073
PortScan,158930
DDoS,128027
DoS GoldenEye,10293
FTP-Patator,7938
SSH-Patator,5897
DoS slowloris,5796
DoS Slowhttptest,5499


## 6. Cleaning steps

Every step records how many rows it removed, so the numbers can go straight into the report.

In [57]:
cleaning_log = []
def log_step(step, rows_before, rows_after, note=""):
    cleaning_log.append({"step": step, "rows_before": rows_before, "rows_after": rows_after,
                         "rows_removed": rows_before - rows_after, "note": note})
    print(f"{step:<45} removed {rows_before - rows_after:>9,} rows  ->  {rows_after:,} left")

feature_cols = [c for c in data.columns if c not in ("Label", "source_file")]

### 6.1 Duplicated column `Fwd Header Length.1`
CIC-IDS2017 contains `Fwd Header Length` twice (pandas renames the second one to `Fwd Header Length.1`). We confirm it is identical and drop it.

In [58]:
if "Fwd Header Length.1" in data.columns:
    same = data["Fwd Header Length"].equals(data["Fwd Header Length.1"])
    print("Fwd Header Length.1 identical to Fwd Header Length:", same)
    data = data.drop(columns=["Fwd Header Length.1"])
    print("Dropped 'Fwd Header Length.1'")
else:
    print("No duplicated 'Fwd Header Length.1' column found")

feature_cols = [c for c in data.columns if c not in ("Label", "source_file")]
print("Feature columns now:", len(feature_cols))

Fwd Header Length.1 identical to Fwd Header Length: True
Dropped 'Fwd Header Length.1'
Feature columns now: 77


### 6.2 Infinite values → NaN
Some flow-rate features (e.g. `Flow Bytes/s`, `Flow Packets/s`) are infinite when a flow lasts 0 microseconds (division by zero during feature extraction).

In [59]:
num = data[feature_cols].select_dtypes(include=[np.number])
inf_counts = np.isinf(num).sum()
inf_counts = inf_counts[inf_counts > 0]
print("Columns with infinite values:")
print(inf_counts.to_string() if len(inf_counts) else "  none")

data[num.columns] = num.replace([np.inf, -np.inf], np.nan)
del num

Columns with infinite values:
Flow Bytes/s      1509
Flow Packets/s    2867


### 6.3 Missing values (NaN)
After the step above, rows with any missing feature value are dropped. They are a tiny fraction of the dataset.

In [60]:
nan_counts = data[feature_cols].isna().sum()
nan_counts = nan_counts[nan_counts > 0]
print("Columns with missing values:")
print(nan_counts.to_string() if len(nan_counts) else "  none")

before = len(data)
data = data.dropna(subset=feature_cols + ["Label"])
log_step("Drop rows with NaN / infinite values", before, len(data))

Columns with missing values:
Flow Bytes/s      2867
Flow Packets/s    2867
Drop rows with NaN / infinite values          removed     2,867 rows  ->  2,827,876 left


### 6.4 Exact duplicate rows
A flow with exactly the same feature values **and** the same label is counted once. The `source_file` column is ignored here so duplicates across days are also caught.

In [61]:
before = len(data)
data = data.drop_duplicates(subset=feature_cols + ["Label"])
log_step("Drop exact duplicate rows", before, len(data))
data = data.reset_index(drop=True)

Drop exact duplicate rows                     removed   329,691 rows  ->  2,498,185 left


### 6.5 Constant columns (report only)
Columns with a single unique value carry no information for a model. They are **listed, not dropped** — the team decides during feature selection.

In [62]:
constant_cols = [c for c in feature_cols if c in data.columns and data[c].nunique(dropna=False) <= 1]
print(f"{len(constant_cols)} constant columns:")
for c in constant_cols:
    print("  -", c, "=", data[c].iloc[0])

8 constant columns:
  - Bwd PSH Flags = 0
  - Bwd URG Flags = 0
  - Fwd Avg Bytes/Bulk = 0
  - Fwd Avg Packets/Bulk = 0
  - Fwd Avg Bulk Rate = 0
  - Bwd Avg Bytes/Bulk = 0
  - Bwd Avg Packets/Bulk = 0
  - Bwd Avg Bulk Rate = 0


### 6.6 Negative values (report only)
Some features (e.g. `Init_Win_bytes_forward`) use `-1` to mean "not available". This is a known quirk of CIC-IDS2017. We only report it here; handling is decided in preprocessing.

In [63]:
neg = (data[feature_cols].select_dtypes(include=[np.number]) < 0).sum()
neg = neg[neg > 0].sort_values(ascending=False)
print("Columns containing negative values (count of rows):")
print(neg.to_string() if len(neg) else "  none")

Columns containing negative values (count of rows):
Init_Win_bytes_backward    1194304
Init_Win_bytes_forward      892949
Flow IAT Min                  2880
Flow Duration                  107
Flow Packets/s                 107
Flow IAT Mean                  107
Flow IAT Max                   107
Flow Bytes/s                    78
min_seg_size_forward            35
Fwd Header Length               35
Bwd Header Length               22
Fwd IAT Min                     17


## 7. Labels for the pipeline
- `Label` — original multi-class label (used by the **DL attack-type classifier**)
- `binary_label` — `0` = BENIGN, `1` = any attack (used by the **ML anomaly detector**)

In [64]:
data["binary_label"] = (data["Label"] != "BENIGN").astype(np.int8)
data["Label"] = data["Label"].astype("category")
data["source_file"] = data["source_file"].astype("category")

print(data["binary_label"].value_counts().rename({0: "BENIGN (0)", 1: "ATTACK (1)"}).to_string())

binary_label
BENIGN (0)    2072444
ATTACK (1)     425741


## 8. Before vs after — label distribution

In [65]:
labels_after = data["Label"].value_counts()
summary = pd.DataFrame({"before": labels_before, "after": labels_after}).fillna(0).astype(int)
summary["removed"] = summary["before"] - summary["after"]
summary["% of clean data"] = (summary["after"] / summary["after"].sum() * 100).round(4)
summary = summary.sort_values("after", ascending=False)
summary.loc["TOTAL"] = [summary["before"].sum(), summary["after"].sum(), summary["removed"].sum(), 100.0]
summary = summary.astype({"before": int, "after": int, "removed": int})
summary

,before,after,removed,% of clean data
Label,,,,
BENIGN,2273097,2072444,200653,82.9580
DoS Hulk,231073,172846,58227,6.9189
DDoS,128027,128014,13,5.1243
PortScan,158930,90694,68236,3.6304
DoS GoldenEye,10293,10286,7,0.4117
FTP-Patator,7938,5931,2007,0.2374
DoS slowloris,5796,5385,411,0.2156
DoS Slowhttptest,5499,5228,271,0.2093
SSH-Patator,5897,3219,2678,0.1289


In [66]:
log_df = pd.DataFrame(cleaning_log)
total_read = sum(r["rows_read"] for r in load_log)
empty = sum(r["empty_rows_dropped"] for r in load_log)
log_df = pd.concat([pd.DataFrame([{"step": "Drop fully empty rows (at load)", "rows_before": total_read,
                                   "rows_after": total_read - empty, "rows_removed": empty, "note": ""}]),
                    log_df], ignore_index=True)
log_df

,step,rows_before,rows_after,rows_removed,note
0,Drop fully empty rows (at load),2830743,2830743,0,
1,Drop rows with NaN / infinite values,2830743,2827876,2867,
2,Drop exact duplicate rows,2827876,2498185,329691,


## 9. Final checks and save

In [67]:
num_cols = data.select_dtypes(include=[np.number]).columns
assert data[feature_cols].isna().sum().sum() == 0, "NaN values remain"
assert not np.isinf(data[num_cols].to_numpy(dtype=np.float64)).any(), "Infinite values remain"
assert not data.duplicated(subset=feature_cols + ["Label"]).any(), "Duplicates remain"
print("All checks passed")

data.to_parquet(OUT_FILE, index=False)
print(f"Saved {OUT_FILE}  ({OUT_FILE.stat().st_size / 1e6:.1f} MB)")
print(f"Final shape: {data.shape[0]:,} rows x {data.shape[1]} columns")

All checks passed
Saved d:\Engineering\SEM 5\MINI PROJECT II\Cybersentry\data\processed\cicids2017_clean.parquet  (254.4 MB)
Final shape: 2,498,185 rows x 80 columns


In [68]:
# Sanity check: reload the saved file
check = pd.read_parquet(OUT_FILE)
print(check.shape)
print(check.dtypes.value_counts().to_string())
del check

(2498185, 80)
int32       27
float32     24
int8        19
int16        7
int64        1
category     1
category     1


## 10. Summary

**Raw data:** 8 CSV files (CIC-IDS2017, MachineLearningCSV version), **2,830,743 flows**, 79 columns (78 features + `Label`).

**Cleaning steps:**

| Step | Rows removed | Rows left |
|---|---:|---:|
| Drop fully empty rows | 0 | 2,830,743 |
| Drop rows with NaN / infinite values | 2,867 | 2,827,876 |
| Drop exact duplicate rows | 329,691 | **2,498,185** |

- Total removed: **332,558 rows (~11.7%)**, almost all of them exact duplicates.
- Dropped the duplicated column `Fwd Header Length.1` (identical to `Fwd Header Length`).
- Fixed the garbled Web Attack labels → `Web Attack - Brute Force`, `Web Attack - XSS`, `Web Attack - Sql Injection`.
- Constant columns are listed in section 6.5 and will be dropped during feature selection.

**Final dataset:** **2,498,185 rows × 80 columns** (77 features + `Label` + `source_file` + `binary_label`), saved as `data/processed/cicids2017_clean.parquet`.

**Labels:** 15 classes (BENIGN + 14 attack types) for the DL stage, plus `binary_label` (0 = BENIGN, 1 = attack) for the ML anomaly-detection stage.

### The imbalance problem (in plain words)
Most of the traffic is normal: roughly 8 out of 10 flows are BENIGN, while some attacks are extremely rare (Heartbleed, Infiltration and SQL Injection have only a few dozen samples each). A model that always predicts "BENIGN" would still get ~80% accuracy while catching zero attacks, so accuracy alone is misleading. We will evaluate with precision, recall, F1, FPR/FNR and confusion matrices per class, and handle rare classes later (class weights, resampling on the training set only, or grouping rare classes). This will be decided with the team in the modelling stage.